# IR Project 2026 - Stable Colab Training Notebook

هذا النوتبوك مصمم حتى يقلل التجميد في Colab.

الفكرة:

- نعمل clone للمشروع من GitHub.
- نربط Google Drive.
- نخزن cache الخاص بـ `ir_datasets` على Drive حتى لا يعيد تحميل Dataset كل مرة.
- نشغل التدريب ونحوّل الخرج إلى `prepare.log` بدل طباعته كله على الشاشة.
- نراقب التقدم بخلايا `tail` خفيفة.
- في النهاية نحفظ `artifacts` و `reports` على Drive.

Dataset النهائية:

`clinicaltrials/2017/trec-pm-2017`

وهي Dataset كاملة فيها أكثر من 200K وثيقة وفيها qrels.

## 0. Runtime Settings

اختاري من Colab:

`Runtime` → `Change runtime type`

اختاري `T4 GPU` إذا متاح. إذا لم يتاح، `CPU` يمشي لكن قد يكون أبطأ.

إذا صار runtime متعب أو علق، اعملي:

`Runtime` → `Restart runtime`

ثم شغلي النوتبوك من البداية.

In [ ]:
import os, sys, platform
print('Python:', sys.version)
print('Platform:', platform.platform())
print('Current directory:', os.getcwd())

## 1. Clone Project From GitHub

هذه الخلية ضرورية كل مرة يبدأ runtime جديد، لأن `/content` مؤقت.

In [ ]:
%cd /content
!rm -rf ir_project
!git clone https://github.com/khaderaldiwani/ir_project.git
%cd /content/ir_project
!git log --oneline -3
!ls -la

## 2. Mount Google Drive

نستخدم Drive لحفظ:

- Dataset cache
- artifacts النهائية
- reports النهائية

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Environment Variables

مهم جداً: نخلي `IR_DATASETS_HOME` على Google Drive حتى لا يعيد التحميل كل مرة.

In [ ]:
import os

os.environ['PYTHONPATH'] = 'src'
os.environ['IR_DATASETS_HOME'] = '/content/drive/MyDrive/ir_datasets_cache'
os.environ['TMP'] = '/content/ir_project/.tmp'
os.environ['TEMP'] = '/content/ir_project/.tmp'

!mkdir -p /content/drive/MyDrive/ir_datasets_cache
!mkdir -p /content/ir_project/.tmp
!echo IR_DATASETS_HOME=$IR_DATASETS_HOME
!echo PYTHONPATH=$PYTHONPATH

## 4. Install Dependencies

`sentence-transformers` مطلوب فقط لتجربة BERT reranking. إذا أخذ وقتاً، يمكن إكمال المشروع بدونه وتشغيل BM25/Hybrid/RAG.

In [ ]:
!pip -q install -r requirements.txt
!pip -q install sentence-transformers

## 5. Verify Dataset

هذه الخلية فقط للتأكد من معلومات Dataset. إذا بدأ تنزيل صغير هنا فهذا طبيعي.

In [ ]:
import os
os.environ['IR_DATASETS_HOME'] = '/content/drive/MyDrive/ir_datasets_cache'

import ir_datasets
dataset_id = 'clinicaltrials/2017/trec-pm-2017'
ds = ir_datasets.load(dataset_id)
print('Dataset:', dataset_id)
print('Docs:', ds.docs_count())
print('Queries:', ds.queries_count())
print('Qrels:', ds.qrels_count())
print('Doc fields:', ds.docs_cls()._fields)
print('Query fields:', ds.queries_cls()._fields)

## 6. Full Training - Stable Mode

هذه أهم خلية.

لا تطبع كل شيء على الشاشة. تكتب الخرج داخل:

`/content/ir_project/prepare.log`

مهم: لا توقفيها طالما الخلية تعمل. قد تستغرق وقتاً.

In [ ]:
%%bash
cd /content/ir_project
export PYTHONPATH=src
export IR_DATASETS_HOME=/content/drive/MyDrive/ir_datasets_cache
export TMP=/content/ir_project/.tmp
export TEMP=/content/ir_project/.tmp

rm -f prepare.log
python -u scripts/prepare.py \
  --dataset clinicaltrials/2017/trec-pm-2017 \
  --max-docs 0 \
  --max-queries 0 \
  --embedding-dims 64 \
  --max-features 30000 \
  --min-df 2 \
  --max-df 0.95 \
  > prepare.log 2>&1

echo 'TRAINING_FINISHED'
tail -n 40 prepare.log

## 7. Check Training Log

إذا شعرتِ أن التدريب متوقف، شغلي هذه الخلية في خلية منفصلة أو بعد انتهاء الخلية السابقة لرؤية آخر الأسطر.

إذا ظهرت `Saving index to ...` أو JSON metadata، فهذا ممتاز.

In [ ]:
!tail -n 60 /content/ir_project/prepare.log || true

## 8. Verify Artifacts

لازم يظهر:

- `documents.sqlite`
- `search_index.joblib`
- `dataset_metadata.json`

In [ ]:
!ls -lh /content/ir_project/artifacts
!test -f /content/ir_project/artifacts/search_index.joblib && echo 'search_index OK'
!test -f /content/ir_project/artifacts/documents.sqlite && echo 'documents.sqlite OK'
!cat /content/ir_project/artifacts/dataset_metadata.json

## 9. If Training Still Fails: Lighter Full-Dataset Build

شغلي هذه الخلية فقط إذا فشلت خلية التدريب الأساسية بسبب الذاكرة.

هذه لا تجزئ Dataset. تستخدم كل الوثائق، لكنها تقلل features وembedding dims.

In [ ]:
# Run only if the main training cell fails due to RAM.
# %%bash
# cd /content/ir_project
# export PYTHONPATH=src
# export IR_DATASETS_HOME=/content/drive/MyDrive/ir_datasets_cache
# export TMP=/content/ir_project/.tmp
# export TEMP=/content/ir_project/.tmp
# rm -f prepare.log
# python -u scripts/prepare.py \
#   --dataset clinicaltrials/2017/trec-pm-2017 \
#   --max-docs 0 \
#   --max-queries 0 \
#   --embedding-dims 32 \
#   --max-features 15000 \
#   --min-df 3 \
#   --max-df 0.90 \
#   > prepare.log 2>&1
# tail -n 40 prepare.log

## 10. Evaluate Base Models

In [ ]:
%%bash
cd /content/ir_project
export PYTHONPATH=src
export IR_DATASETS_HOME=/content/drive/MyDrive/ir_datasets_cache

python -u scripts/evaluate.py --dataset clinicaltrials/2017/trec-pm-2017 --max-queries 0 > evaluate.log 2>&1
tail -n 60 evaluate.log
cat artifacts/evaluation_metrics.csv

## 11. Evaluate With Query Refinement

In [ ]:
%%bash
cd /content/ir_project
export PYTHONPATH=src
export IR_DATASETS_HOME=/content/drive/MyDrive/ir_datasets_cache

python -u scripts/evaluate.py --dataset clinicaltrials/2017/trec-pm-2017 --max-queries 0 --refine > evaluate_refined.log 2>&1
tail -n 60 evaluate_refined.log
cat artifacts/evaluation_metrics_refined.csv

## 12. Test Search

In [ ]:
!PYTHONPATH=src IR_DATASETS_HOME=/content/drive/MyDrive/ir_datasets_cache python scripts/search.py "lung cancer EGFR adult" --method bm25 --top-k 5
!PYTHONPATH=src IR_DATASETS_HOME=/content/drive/MyDrive/ir_datasets_cache python scripts/search.py "breast cancer treatment" --method hybrid_parallel --top-k 5

## 13. Optional BERT Reranking

إذا أخذت وقتاً أو فشلت بسبب الإنترنت، اتركيها. المشروع الأساسي يعمل بدونها.

In [ ]:
!PYTHONPATH=src python scripts/download_bert_model.py
!PYTHONPATH=src python scripts/search.py "lung cancer EGFR adult" --method bert_rerank --top-k 5

## 14. Save Artifacts And Reports To Google Drive

هذه الخطوة تحفظ النتائج النهائية لتنزيلها على الجهاز المحلي.

In [ ]:
!rm -rf /content/drive/MyDrive/ir_project_saved
!mkdir -p /content/drive/MyDrive/ir_project_saved
!cp -r /content/ir_project/artifacts /content/drive/MyDrive/ir_project_saved/
!cp -r /content/ir_project/reports /content/drive/MyDrive/ir_project_saved/
!find /content/drive/MyDrive/ir_project_saved -maxdepth 2 -type f | head -80

## 15. What To Do Locally

بعد تنزيل `artifacts` و `reports` من Drive، استبدلي المجلدين داخل المشروع المحلي.

ثم شغلي:

```powershell
cd "C:\\Users\\Lenovo\\Desktop\\ir dociment\\ir_project"
.\\run_app.ps1
```

وافتحي:

```text
http://localhost:8501
```